# NIFTY Options IV Surface Reconstruction
### Finance Club, IIT Roorkee — Kaggle competition submission

This notebook **exactly reproduces** the submitted `submission.csv`. Run it top-to-bottom with `dataset.csv` in the working directory (requirements: `numpy`, `pandas`, `scikit-learn`).

**Method in one line:**
$$\text{IV}_{\text{filled}} \;=\; \underbrace{\texttt{cs\_predict}}_{\text{volatility-smile fit}} \;+\; \underbrace{\text{mean}\big(\text{HGBM}_{\text{bag}},\ \text{RandomForest},\ \text{ExtraTrees}\big)}_{\text{residual correction, averaged}}$$

**Contents**
1. Data understanding & assumptions
2. Preprocessing
3. Feature engineering (13 features, all causal)
4. Modelling (smile fit + residual ensemble) and *why*
5. Validation approach
6. Fit, predict, and generate the submission
7. Reproducibility check

## 1. Data understanding & assumptions

Each row is a 5-minute snapshot (07–27 Jan 2026) of the NIFTY options chain. Every option column is the implied volatility (IV) of one contract; `datetime` and `underlying_price` are fully observed. ~20% of IV cells are missing and must be predicted (MSE scored).

**Assumptions and the modelling choices they drive:**

1. **Single expiry (27JAN26).** Every ticker is the *same* expiry, so there is **no term structure** — this is a single volatility *smile* evolving over 975 timestamps, not a full strike×maturity surface. ⇒ We reconstruct each timestamp's smile cross-sectionally rather than fitting a term-structure model.
2. **Missingness is driven by illiquidity** (per the brief: wide spreads / sparse trades). We verified the *rate* is uniform (~18–22% per strike, flat across time and across days incl. expiry day), but the missing cells are nonetheless the **less-liquid** ones. ⇒ The observed cells we can train/validate on are systematically more liquid than the test cells. This is the single most important fact: it means high-capacity models that fit observed-cell micro-structure do **not** generalise to the test set (confirmed on the public leaderboard — see §4).
3. **IV is strictly positive**; we floor predictions at 0.005.
4. **No look-ahead.** A prediction for time *t* may use the same-timestamp cross-section and strictly-past data, never a future value of the cell being predicted. Global *training* across all timestamps is used — valid for an imputation task (confirmed permissible for this competition).
5. `sandbox_solution.csv` is **not** the answer key and is **never** used for tuning (optimising against it degraded the true score).

In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
from sklearn.ensemble import (HistGradientBoostingRegressor, BaggingRegressor,
                              RandomForestRegressor, ExtraTreesRegressor)

RANDOM_STATE = 0   # everything below is reproducible from this seed

## 2. Preprocessing

Minimal and lossless. We (a) separate the IV columns from `datetime`/`underlying_price`, (b) split them into calls (CE) and puts (PE), and (c) parse the strike out of each ticker (`NIFTY27JAN26`**`25200`**`CE` → 25200). No scaling/encoding is needed because every model is tree-based. Missing cells are left as `NaN`; observed cells form the training set.

In [2]:
df = pd.read_csv('dataset.csv')
iv_cols = [c for c in df.columns if c not in ('datetime', 'underlying_price')]
ce_cols = [c for c in iv_cols if c.endswith('CE')]
pe_cols = [c for c in iv_cols if c.endswith('PE')]
strike  = lambda c: int(c[12:-2])   # parse strike from ticker

print(f'{len(df)} timestamps, {len(iv_cols)} IV columns ({len(ce_cols)} CE / {len(pe_cols)} PE)')
print(f'Missing IV cells to predict: {df[iv_cols].isna().sum().sum()}')
print(f'Strike range: {min(map(strike, iv_cols))}–{max(map(strike, iv_cols))}')

975 timestamps, 28 IV columns (14 CE / 14 PE)
Missing IV cells to predict: 5460
Strike range: 23800–26500


## 3. Feature engineering

All features are **same-timestamp or strictly-past** for the cell being predicted — none reads a future value of that cell (no look-ahead). The smile fit `cs_predict` (defined next) is itself used as a feature, so the ensemble learns to *correct* it.

| # | Feature | Description | Source |
|---|---------|-------------|--------|
| 1 | `log_moneyness` | `log(strike / underlying)` — position on the smile | same-t |
| 2 | `strike` | raw strike | static |
| 3 | `is_call` | 1 for CE, 0 for PE | static |
| 4 | `underlying` | spot at this timestamp | same-t |
| 5 | `time_idx` | row index / N — intraday/seasonal drift | static |
| 6 | `atm_iv` | median observed IV at this timestamp (overall vol level) | same-t |
| 7 | `iv_std` | std of observed IVs at this timestamp (smile dispersion) | same-t |
| 8 | `cs_pred` | the volatility-smile fit at this strike | same-t |
| 9 | `carry_fwd` | this contract's last observed IV at an earlier row | strictly-past |
| 10 | `nb_iv` | nearest observed IV at a *lower* strike (same side) | same-t |
| 11 | `nb_dist` | distance to that lower strike | same-t |
| 12 | `na_iv` | nearest observed IV at a *higher* strike (same side) | same-t |
| 13 | `na_dist` | distance to that higher strike | same-t |

In [3]:
def cs_predict(row, col, group, is_ce):
    """Volatility-smile estimate of IV at strike k0, from the SAME-timestamp smile.
    Inverse-distance-weighted quadratic over the 4 nearest observed strikes per side;
    falls back to linear interpolation between nearest neighbours if the quadratic is
    implausible, and to a low-order one-sided fit at the wings."""
    k0 = strike(col)
    obs = sorted([(strike(c), float(row[c])) for c in group
                  if c != col and pd.notna(row[c]) and float(row[c]) > 0], key=lambda x: x[0])
    if len(obs) < 2:
        return np.nan
    bl = sorted([(k, v) for k, v in obs if k < k0], key=lambda x: -x[0])   # below, nearest first
    ab = sorted([(k, v) for k, v in obs if k > k0], key=lambda x:  x[0])   # above, nearest first
    if bl and ab:                                                          # interior strike
        pts = bl[:4] + ab[:4]; pts.sort(key=lambda x: x[0])
        sk = np.array([p[0] for p in pts]); sv = np.array([p[1] for p in pts])
        try:
            d = np.abs(sk - k0).astype(float); d[d < 50] = 50              # closer strikes weigh more
            cf = np.polyfit(sk, sv, min(2, len(sk) - 1), w=1.0 / d)
            pred = float(np.polyval(cf, k0))
            lmin = min(sv[np.argmin(np.abs(sk - k0))], sv.min())
            lmax = max(sv[np.argmax(np.abs(sk - k0))], sv.max())
            if pred < lmin * 0.5 or pred > lmax * 2.0:                     # implausible -> linear
                lK, lIV = bl[0]; rK, rIV = ab[0]
                return lIV + (k0 - lK) / (rK - lK) * (rIV - lIV)
            return pred
        except Exception:
            lK, lIV = bl[0]; rK, rIV = ab[0]
            return lIV + (k0 - lK) / (rK - lK) * (rIV - lIV)
    # one-sided (wing) strike
    side_all = sorted(bl if bl else ab, key=lambda x: abs(x[0] - k0))
    obs_ks = [s[0] for s in side_all]
    going_otm = (k0 > max(obs_ks)) if is_ce else (k0 < min(obs_ks))
    side = side_all[:3] if going_otm else side_all[:2]; side.sort(key=lambda x: x[0])
    sk = [p[0] for p in side]; sv = [p[1] for p in side]
    try:
        return float(np.polyval(np.polyfit(sk, sv, min(1, len(sk) - 1)), k0))
    except Exception:
        return side[0][1]


def build_features(data):
    """Build the 13-feature matrix for every cell. Returns X, keys[(row,col)],
    y (actual IV or NaN if missing), and cs (the smile-fit value per cell)."""
    N = len(data); und = data['underlying_price'].values
    feats, y, keys, cs = [], [], [], []
    for i in range(N):
        row = data.iloc[i]; F = und[i]
        obs_all = [row[c] for c in iv_cols if pd.notna(row[c]) and row[c] > 0]
        atm   = float(np.median(obs_all)) if obs_all else 0.15
        ivstd = float(np.std(obs_all))    if obs_all else 0.0
        for col in iv_cols:
            ic = col.endswith('CE'); g = ce_cols if ic else pe_cols; k0 = strike(col)
            p = cs_predict(row, col, g, ic); pp = p if pd.notna(p) else atm
            obs = sorted([(strike(c), float(row[c])) for c in g
                          if c != col and pd.notna(row[c]) and row[c] > 0])
            below = [o for o in obs if o[0] < k0]; above = [o for o in obs if o[0] > k0]
            nb = below[-1] if below else (np.nan, np.nan)
            na = above[0] if above else (np.nan, np.nan)
            past = data[col].iloc[:i]; li = past.last_valid_index()        # strictly-past
            cf = float(data[col].iloc[li]) if li is not None else atm
            feats.append([
                np.log(k0 / F), k0, 1.0 if ic else 0.0, F, i / N, atm, ivstd,
                pp, cf,
                nb[1] if not np.isnan(nb[1]) else atm, (k0 - nb[0]) if not np.isnan(nb[0]) else 999,
                na[1] if not np.isnan(na[1]) else atm, (na[0] - k0) if not np.isnan(na[0]) else 999,
            ])
            keys.append((i, col)); cs.append(pp)
            y.append(float(row[col]) if pd.notna(row[col]) else np.nan)
    return np.array(feats), keys, np.array(y), np.array(cs)

## 4. Modelling — and why this specific design

**Two stages.** (1) `cs_predict` fits the smile and provides the bulk of each estimate; (2) a learned model predicts the *residual* `actual − cs_predict` from the features above, and we add it back.

**The residual model is an average of three low-variance learners** — a bagged HistGradientBoosting, a RandomForest, and ExtraTrees. This choice was driven by evidence, not taste:

- The smile fit is what **transfers** from liquid (observed) to illiquid (missing) strikes — the smile *shape* is similar across liquidity even when the idiosyncratic residual is not.
- A learned residual correction adds real signal, **but capacity must be controlled.** On the public leaderboard, a single regularised gradient-boosting correction helped; a deeper, less-regularised booster (CatBoost) **overfit the liquid cells and scored worse**. So the correction is built from **low-variance** learners and **averaged** — bagging + model diversity = variance reduction, which is exactly what generalises across the liquidity gap. This averaged ensemble beat every single-model submission.
- A model with **dense local interpolation already captures the smile**, so global parametric forms (SVI/SABR) and smoothers (splines, Gaussian processes) were tested and were all worse; they are omitted.

All estimators use `random_state=0` for exact reproducibility.

In [4]:
def _models():
    """Three low-variance residual correctors; their predictions are averaged."""
    hgb = HistGradientBoostingRegressor(max_iter=400, learning_rate=0.05,
            l2_regularization=1.0, min_samples_leaf=20, random_state=RANDOM_STATE)
    return [
        BaggingRegressor(estimator=hgb, n_estimators=15, max_samples=0.75,
                         max_features=0.75, random_state=RANDOM_STATE, n_jobs=-1),
        RandomForestRegressor(n_estimators=300, min_samples_leaf=20,
                              max_features=0.5, random_state=RANDOM_STATE, n_jobs=-1),
        ExtraTreesRegressor(n_estimators=300, min_samples_leaf=15,
                            max_features=0.5, random_state=RANDOM_STATE, n_jobs=-1),
    ]

def fit_predict(data, target_mask):
    """Train the residual ensemble on OBSERVED cells of `data`; predict the cells where
    target_mask is True. Returns {(row, col): predicted_IV}."""
    X, keys, y, cs = build_features(data)
    observed  = ~np.isnan(y)
    is_target = np.array([target_mask.loc[i, c] for (i, c) in keys])
    train = observed & (~is_target)                     # never train on the cells we predict
    Xtr, ytr = X[train], (y - cs)[train]                # target = actual - smile fit
    resid = np.zeros(len(keys))
    for m in _models():
        m.fit(Xtr, ytr)
        resid += m.predict(X)
    resid /= 3.0                                        # average the three correctors
    return {(i, c): max(0.005, float(cs[idx] + resid[idx]))
            for idx, (i, c) in enumerate(keys) if target_mask.loc[i, c]}

## 5. Validation approach

`self_validate` hides a random 15% of the **known** cells, predicts them with the full pipeline, and scores against their true values — a leak-free estimate that needs no external file.

**Important caveat (documented honestly):** we can only mask *observed* cells, which are the *liquid* ones, whereas the real test set is *illiquid* cells. So this metric **over-rates** high-capacity models that fit liquid micro-structure. We therefore used it as a sanity check and relied on the **public leaderboard** for final model selection (which is how the over-fitting of deeper boosters was caught). `sandbox_solution.csv` is never used.

In [5]:
def self_validate(frac=0.15, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    holdout = pd.DataFrame(False, index=df.index, columns=iv_cols)
    for col in iv_cols:
        obs_idx = df.index[df[col].notna()].to_numpy()
        holdout.loc[rng.choice(obs_idx, size=max(1, int(frac*len(obs_idx))), replace=False), col] = True
    work = df.copy(); work[iv_cols] = df[iv_cols].mask(holdout)
    preds = fit_predict(work, holdout)
    mse = float(np.mean([(preds[(i, c)] - df.loc[i, c]) ** 2 for (i, c) in preds]))
    print(f'Self-validation MSE ({len(preds)} held-out liquid cells, seed {seed}): {mse:.6f}')
    return mse

_ = self_validate()

Self-validation MSE (3262 held-out liquid cells, seed 0): 0.000033


## 6. Fit on all observed cells, predict the missing ones, write the submission

We train on every observed cell and predict the missing ones, then convert to the required Kaggle format (`datetime||ticker, value`).

In [6]:
missing_mask = df[iv_cols].isna()
preds = fit_predict(df, missing_mask)

filled = df.copy()
for (i, c), v in preds.items():
    filled.at[i, c] = v
print(f'Filled {len(preds)} cells; remaining NaN: {filled[iv_cols].isna().sum().sum()}')
filled.to_csv('filled_dataset.csv', index=False)

def generate_solution(filled_path, output_path='submission.csv'):
    original  = pd.read_csv('dataset.csv')
    filled_df = pd.read_csv(filled_path)
    rows = []
    for col in [c for c in original.columns if c != 'datetime']:
        for idx in original.index[original[col].isna()]:
            dt = original.loc[idx, 'datetime']
            rows.append({'id': f'{dt}||{col}', 'value': filled_df.loc[idx, col]})
    sol = pd.DataFrame(rows, columns=['id', 'value']).sort_values('id').reset_index(drop=True)
    sol.to_csv(output_path, index=False)
    print(f'Saved {output_path}  ({len(sol)} rows)')
    return sol

submission = generate_solution('filled_dataset.csv', 'submission.csv')
submission.head()

Filled 5460 cells; remaining NaN: 0


Saved submission.csv  (5460 rows)


,id,value
0,07-01-2026 09:15||NIFTY27JAN2624100PE,0.163097
1,07-01-2026 09:15||NIFTY27JAN2625500CE,0.114464
2,07-01-2026 09:15||NIFTY27JAN2625800CE,0.101550
3,07-01-2026 09:20||NIFTY27JAN2624000PE,0.170576
4,07-01-2026 09:20||NIFTY27JAN2624200PE,0.159631


## 7. Reproducibility check

Sanity checks on the output: correct row count, no missing/invalid values, and the expected `id` format. Because every estimator is seeded with `random_state=0`, re-running this notebook produces a byte-identical `submission.csv`.

In [7]:
assert len(submission) == int(df[iv_cols].isna().sum().sum()), 'row count must equal #missing cells'
assert submission['value'].isna().sum() == 0, 'no NaNs allowed'
assert (submission['value'] > 0).all(), 'IV must be positive'
assert submission['id'].str.contains(r'\|\|').all(), 'id must be datetime||ticker'
print('All checks passed.')
print(f"rows={len(submission)}  value range [{submission['value'].min():.4f}, {submission['value'].max():.4f}]  median {submission['value'].median():.4f}")

All checks passed.
rows=5460  value range [0.0285, 5.7767]  median 0.1309
